In [1]:
import numpy as np
import matplotlib.pyplot as plt
import utils
import ot
import scipy
import os

%load_ext autoreload
%autoreload 2

In [ ]:
baseline_accs = np.zeros((5, 10))
fgw_accs = np.zeros((5, 10))

k = 10
alpha = 0.5

dataset_path = "datasets/action_smplx_models"

for i in range(5):
    path = dataset_path + "/" + np.random.choice(os.listdir(dataset_path))
    randidx = np.random.randint(0, int(np.load(path)["mocap_time_length"] * 120) - 105)
    points1, faces1 = utils.sampled_verts_from_path(path, idx = randidx, return_faces=True)
    C1 = utils.graph_distance_within_cloud_minimally_connected(points1, k = k)
    for j in range(np.arange(10, 101, 10).shape[0]):
        td = np.arange(10,101,5)[j]
        points2, faces2 = utils.sampled_verts_from_path(path, idx = randidx + td, return_faces=True)
        a = np.ones(1000)/1000
        b = np.ones(1000)/1000
        M = ot.dist(points1, points2)
        G1 = ot.solve(M, a, b).plan
        baseline_accs[i,j] = utils.region_accuracy_adjusted(G1, faces1, faces2)
        C2 = utils.graph_distance_within_cloud_minimally_connected(points2, k = k)
        
        G2 = ot.fused_gromov_wasserstein(M, C1, C2, alpha = alpha)

        fgw_accs[i,j] = utils.region_accuracy_adjusted(G2, faces1, faces2)
        print(i,j,"done")

In [ ]:
plt.plot(np.arange(10, 101, 10), baseline_accs.mean(axis = 0), label = "baseline")
plt.plot(np.arange(10, 101, 10), fgw_accs.mean(axis = 0), label = "fgw")
plt.legend()
plt.title("fgw alpha = 0.5, k = 10")
plt.xlabel("time delta")
plt.ylabel("adjusted accuracy")